# Задача 1. Индекс цен по конкурентам и предложения по скидкам/политикам исходя из стратегии

Для анализа использованы листы «Данные для задачи 1» и «Справочник наценок».

**Допущения:**

— правила наценок применяются на 2.06.2025, поскольку дата мониторинга не указана;

— встать в минимальную цену на рынке тарктуется как стать равными минимальной цене на рынке.

**Методика:** 

Цена до скидки = закупка × (1 + наценка из справочника). 

Индекс цен = наша цена / цена конкурента. 

Индекс — простое среднее по SKU.

# Аналитическая записка

**1. Индекс цен.** 
Мы дороже почти всех конкурентов (средний индекс 1,19–1,32), кроме tsum.ru, где мы на уровне рынка (1,01).
Наибольший разрыв с Ozon (1,32) и Wildberries (1,30).

**2. Результат расчёта скидок.**
- В минимум рынка встаём по 527 из 683 SKU (77%). 
Из них по 13 SKU цена уже в рынке или ниже, по 514 нужна скидка до цены конкурента.

- По 156 SKU (23%) цель недостижима из-за ограничения по минимальной марже 10%.

**3. Рекомендации.**
Пересмотреть справочник наценок: скидки, необходимые для соблюдения стратегии, в среднем около 22%, то есть базовые цены сильно выше рынка. Наценки нужно приблизить к рыночным, тогда скидки будут точечными. 

In [1]:
import pandas as pd
import numpy as np

date = pd.Timestamp('2025-06-02')


file_path = "data/Тестовое_задание_Data_аналитик_ЦО_2025_2.xlsx"

products = pd.read_excel(file_path, sheet_name="Данные для задачи 1")
rules = pd.read_excel(file_path, sheet_name="Справочник наценок")

In [2]:
print(products.dtypes)
display(products.head(3))

Бренд                   object
Код товара               int64
Наименование товара     object
Категория1              object
Категория2              object
Категория3              object
Категория4              object
Сегмент                 object
Цена закупки товара      int64
letu.ru                float64
iledebeaute.ru         float64
rivegauche.ru          float64
Ozon                   float64
Wildberries            float64
MagnitMarket           float64
Megamarket             float64
lamoda.ru              float64
tsum.ru                float64
visagehall.ru          float64
dtype: object


,Бренд,Код товара,Наименование товара,Категория1,Категория2,Категория3,Категория4,Сегмент,Цена закупки товара,letu.ru,iledebeaute.ru,rivegauche.ru,Ozon,Wildberries,MagnitMarket,Megamarket,lamoda.ru,tsum.ru,visagehall.ru
0,Clarins,19000412437,CLARINS Тушь д/ресниц Wonder volume xxl 8мл,Макияж,Глаза,Тушь для ресниц,NaN,Люкс,983,NaN,1029.0,NaN,NaN,NaN,NaN,NaN,1117.0,NaN,NaN
1,Clarins,14037100013,Clarins Флюид тональный устойчивый с омолажива...,Макияж,Лицо,Тональные основы,Жидкие тональные основы,Люкс,1957,NaN,2107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Clarins,19000033387,CLARINS Масло-блеск для губ Lip Comfort Oil т....,Макияж,Губы,Масло для губ_макияж,NaN,Люкс,850,NaN,1065.0,919.0,964.0,NaN,NaN,NaN,924.0,1574.0,964.0


In [3]:
print(rules.dtypes)
display(rules.head(3))

Сегмент рынка             object
Категория                 object
Бренд                     object
Продукт                  float64
Дата начала       datetime64[ns]
Дата окончания    datetime64[ns]
Наценка                  float64
Активный                  object
dtype: object


,Сегмент рынка,Категория,Бренд,Продукт,Дата начала,Дата окончания,Наценка,Активный
0,Люкс,NaN,NaN,NaN,2025-01-01,NaT,1.2,Да
1,Массмаркет,NaN,NaN,NaN,2025-01-01,NaT,0.7,Да
2,Люкс,Макияж,NaN,NaN,2025-01-01,NaT,0.9,Да


In [4]:
# базовая проверка данных
print(f'Дубликаты по коду SKU: {products["Код товара"].duplicated().sum()}')
print(f'Отрицательная или нулевая закупочная цена: {(products["Цена закупки товара"]<=0).sum()}')

competitors = [
    "letu.ru", "iledebeaute.ru", "rivegauche.ru","Ozon", "Wildberries", "MagnitMarket", "Megamarket",
    "lamoda.ru",  "tsum.ru", "visagehall.ru"
]

print(f'Отрицательная или нулевая цена конкурентов: {(products[competitors]<=0).sum().sum()}')
print(f'Отрицательная или нулевая наценка: {(rules["Наценка"]<=0).sum()}')

Дубликаты по коду SKU: 0
Отрицательная или нулевая закупочная цена: 0
Отрицательная или нулевая цена конкурентов: 0
Отрицательная или нулевая наценка: 0


In [5]:
# берем только активные правила
active_rules = rules[
    (rules["Активный"] == "Да") & 
    (rules["Дата начала"] <= date) & 
    ((rules["Дата окончания"] >= date) | (rules["Дата окончания"].isna()))
].copy()


# убедимся в отсутствии дубликатов правил
duplicates = active_rules.duplicated(
    ["Продукт", "Бренд", "Категория", "Сегмент рынка"]
).sum()

print(f'Дубликаты правил: {duplicates}')

Дубликаты правил: 0


In [6]:
# Проверям типы данных в колонка "Код товара" и "Продукт" на сопоставимость

print(products["Код товара"].map(type).value_counts())
print(active_rules["Продукт"].map(type).value_counts())

<class 'int'>    683
Name: Код товара, dtype: int64
<class 'float'>    120
Name: Продукт, dtype: int64


In [7]:
# меняем тип данных на строковый
products["Код товара"] = products["Код товара"].astype(str)
active_rules["Продукт"] = active_rules["Продукт"].fillna(-1).astype(int).astype(str).replace("-1", np.nan)

In [8]:
# делаем копию таблицы продуктов и добавляем столбец для добавления наценок
df = products.copy()
df["Наценка"] = np.nan


# добавляем наценки по правилам

f_rules = [
    (["Код товара"], ["Продукт"]),
    (["Бренд", "Категория3"], ["Бренд", "Категория"]),
    (["Бренд", "Категория2"], ["Бренд", "Категория"]),
    (["Бренд", "Категория1"], ["Бренд", "Категория"]),
    (["Бренд"], ["Бренд"]),
    (["Сегмент", "Категория3"], ["Сегмент рынка", "Категория"]),
    (["Сегмент", "Категория2"], ["Сегмент рынка", "Категория"]),
    (["Сегмент", "Категория1"], ["Сегмент рынка", "Категория"]),
    (["Сегмент"], ["Сегмент рынка"]),
]


def find_markup(df, active_rules, df_keys, rules_keys):
    
    rules = active_rules.dropna(subset=rules_keys)

    if rules_keys != ["Продукт"]:
        rules = rules[
            rules[["Продукт", "Бренд", "Категория", "Сегмент рынка"]].drop(columns=rules_keys).isna().all(axis=1)
        ]

    merge = df.merge(
        rules[rules_keys + ["Наценка"]],
        left_on=df_keys,
        right_on=rules_keys,
        how="left",
        suffixes=("", "_правило")
    )

    merge["Наценка"] = merge["Наценка"].fillna(merge["Наценка_правило"])

    return merge[df.columns]


for rule in f_rules:
    if df["Наценка"].notna().all():
        break
    df = find_markup(df, active_rules, *rule)


# проверяем остались ли товары без наценки
print(f'Позиций без наценки: {df["Наценка"].isna().sum()}')

display(df[["Код товара", "Наценка"]].head(3))

Позиций без наценки: 0


,Код товара,Наценка
0,19000412437,1.03
1,14037100013,1.03
2,19000033387,1.03


In [9]:
# строим индекс цен по конкурентам

df["Цена"] = (df["Цена закупки товара"] * (1 + df["Наценка"])).round(2)

price_index = df[competitors].rdiv(df["Цена"], axis="index")

summary = price_index.agg(["mean", "count"]).T

summary = summary.rename(columns={"mean": "Средний индекс цен", "count": "Количество товаров"}).round(2)

summary.index.name = "Конкурент"

display(summary)

,Средний индекс цен,Количество товаров
Конкурент,,
letu.ru,1.19,193.0
iledebeaute.ru,1.28,495.0
rivegauche.ru,1.23,610.0
Ozon,1.32,471.0
Wildberries,1.30,176.0
MagnitMarket,1.22,120.0
Megamarket,1.19,89.0
lamoda.ru,1.22,364.0
tsum.ru,1.01,226.0


In [10]:
# расчёт предложения по скидке

min_margin = 0.10

df["Мин_цена_конкурентов"] = df[competitors].min(axis=1)

df["Мин_допустимая_цена"] = np.ceil(df["Цена закупки товара"] / (1 - min_margin))

mask = ((df["Мин_цена_конкурентов"] < df["Цена"]) & (df["Цена"] >= df["Мин_допустимая_цена"]))

df["Предлагаемая_скидка_%"] = 0.0

df.loc[mask, "Предлагаемая_скидка_%"] = (
    1
    - df.loc[
        mask, ["Мин_цена_конкурентов", "Мин_допустимая_цена"]
    ].max(axis=1) / df.loc[mask, "Цена"]
) * 100

df["Маржинальность_%"] = (
    1
    - df["Цена закупки товара"]
    / (df["Цена"] * (1 - df["Предлагаемая_скидка_%"] / 100))
) * 100

conditions = [
    df["Цена"] < df["Мин_допустимая_цена"],
    df["Мин_цена_конкурентов"].isna(),
    df["Цена"] <= df["Мин_цена_конкурентов"],
    mask & (
        df["Мин_цена_конкурентов"] >= df["Мин_допустимая_цена"]
    ),
    mask & (
        df["Мин_цена_конкурентов"] < df["Мин_допустимая_цена"]
    ),
]

choices = [
    "база ниже допустимой цены — требуется отдельное решение",
    "нет мониторинга",
    "в рынке / дешевле (скидка не нужна)",
    "скидка до цены конкурента",
    "упор в мин. маржу",
]

df["Статус"] = np.select(conditions, choices, default="ошибка")

# Округление только после расчётов и определения статусов.
df["Предлагаемая_скидка_%"] = df["Предлагаемая_скидка_%"].round(1)
df["Маржинальность_%"] = df["Маржинальность_%"].round(2)

final_cols = [
    "Код товара",
    "Цена закупки товара",
    "Цена",
    "Мин_цена_конкурентов",
    "Мин_допустимая_цена",
    "Предлагаемая_скидка_%",
    "Маржинальность_%",
    "Статус",
]

print(f'Средняя предлагаемая скидка, %: {(df["Предлагаемая_скидка_%"].mean()).round(1)}')
display(df[final_cols].head())
display(df["Статус"].value_counts())

Средняя предлагаемая скидка, %: 22.3


,Код товара,Цена закупки товара,Цена,Мин_цена_конкурентов,Мин_допустимая_цена,Предлагаемая_скидка_%,Маржинальность_%,Статус
0,19000412437,983,1995.49,1029.0,1093.0,45.2,10.06,упор в мин. маржу
1,14037100013,1957,3972.71,2107.0,2175.0,45.3,10.02,упор в мин. маржу
2,19000033387,850,1725.50,919.0,945.0,45.2,10.05,упор в мин. маржу
3,14037100015,2145,4354.35,2355.0,2384.0,45.3,10.03,упор в мин. маржу
4,19000168458,835,1695.05,924.0,928.0,45.3,10.02,упор в мин. маржу


скидка до цены конкурента              514
упор в мин. маржу                      156
в рынке / дешевле (скидка не нужна)     13
Name: Статус, dtype: int64

In [11]:
# Сохраняем индекс цен
summary.to_csv("results/Задача_1_Индекс_цен.csv", sep=";", encoding="utf-8-sig")

# Сохраняем предложения по скидкам
df[final_cols].to_csv("results/Задача_1_Предложение_по_скидкам.csv", index=False, sep=";", encoding="utf-8-sig")

print("✅ Файлы успешно сохранены в папку 'results'")

✅ Файлы успешно сохранены в папку 'results'
